In [0]:
#Config

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)   # fixed seed = same data every run

N_DAYS = 60
START_DATE = pd.Timestamp("2026-07-01")
N_CUSTOMERS = 20000

In [0]:
#Creating the layer schemas

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")

In [0]:
#Stores and daily context

stores = pd.DataFrame({
    "store_id": [f"DS0{i}" for i in range(1, 7)],
    "store_name": [f"Dark Store {i}" for i in range(1, 7)],
    "zone": ["North", "North", "South", "South", "East", "West"],
    "base_daily_orders": [320, 280, 350, 260, 300, 290],  # how busy each store's area is
})

dates = pd.date_range(START_DATE, periods=N_DAYS, freq="D")
daily_context = pd.DataFrame({
    "date": dates.date,
    "is_weekend": dates.dayofweek >= 5,
    "is_rainy": rng.random(N_DAYS) < 0.20,   # roughly 1 rainy day in 5
})

In [0]:
#SKUs

categories = [
    # name, number of SKUs, is_perishable, typical price (INR)
    ("Fruits & Vegetables", 60, True, 60),
    ("Dairy & Eggs", 35, False, 70),
    ("Bakery", 20, False, 50),
    ("Snacks", 50, False, 40),
    ("Beverages", 40, False, 60),
    ("Staples", 35, False, 150),
    ("Personal Care", 30, False, 180),
    ("Household", 30, False, 120),
]

rows, n = [], 1
for name, count, perishable, typical_price in categories:
    for _ in range(count):
        rows.append({
            "sku_id": f"SKU{n:04d}",
            "category": name,
            "is_perishable": perishable,
            "unit_price": int(max(10, rng.lognormal(np.log(typical_price), 0.4))),
            "shelf_life_days": int(rng.integers(2, 6)) if perishable else 180,
            "popularity": float(rng.pareto(1.5) + 1),   # heavy tail: few SKUs dominate
        })
        n += 1
skus = pd.DataFrame(rows)

In [0]:
#Customers

w = stores["base_daily_orders"] / stores["base_daily_orders"].sum()
signup = START_DATE + pd.to_timedelta(rng.integers(-400, N_DAYS, N_CUSTOMERS), unit="D")

customers = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N_CUSTOMERS + 1)],
    "signup_date": signup,
    "home_store_id": rng.choice(stores["store_id"], N_CUSTOMERS, p=w),
})
customers["signup_date"] = customers["signup_date"].dt.date

In [0]:
#Save to Bronze

def save_bronze(pdf, name):
    sdf = spark.createDataFrame(pdf)
    sdf.write.mode("overwrite").saveAsTable(f"workspace.bronze.{name}")
    print(f"{name}: {sdf.count()} rows")

save_bronze(stores, "stores")
save_bronze(skus, "skus")
save_bronze(customers, "customers")
save_bronze(daily_context, "daily_context")

In [0]:
%sql
--Check

SELECT category, COUNT(*) AS skus, ROUND(AVG(unit_price)) AS avg_price
FROM workspace.bronze.skus
GROUP BY category
ORDER BY skus DESC;

In [0]:
#Demand settings

HOURS = np.arange(6, 24)   # stores operate 06:00-23:59
HOUR_WEIGHTS = np.array([0.5, 1.5, 2.5, 3, 3, 4, 5, 5, 4, 3.5, 4, 5, 6.5, 8.5, 9.5, 9, 6, 3])
HOUR_P = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()   # lunch peak and a bigger dinner peak
assert len(HOURS) == len(HOUR_WEIGHTS)

WEEKEND_MULT = 1.15   # weekends are busier
RAIN_MULT = 1.10      # rain increases orders slightly

In [0]:
#who orders(customer pools)

cust_ids = customers["customer_id"].values
cust_weight = rng.gamma(0.7, 1.0, N_CUSTOMERS) + 0.05   # heavy tail: a few customers order a lot
cust_signup = pd.to_datetime(customers["signup_date"]).values

# customers grouped by their home store
home_idx = {s: np.where(customers["home_store_id"].values == s)[0] for s in stores["store_id"]}

In [0]:
#Generate orders, day by day and store by store

frames = []
for day, ctx in zip(dates, daily_context.itertuples()):
    for s in stores.itertuples():
        mult = (WEEKEND_MULT if ctx.is_weekend else 1.0) * (RAIN_MULT if ctx.is_rainy else 1.0)
        n = rng.poisson(s.base_daily_orders * mult)

        # only customers of this store who have already signed up can order
        pool = home_idx[s.store_id]
        pool = pool[cust_signup[pool] <= day.to_datetime64()]
        p = cust_weight[pool] / cust_weight[pool].sum()

        chosen = rng.choice(pool, n, p=p)
        hour = rng.choice(HOURS, n, p=HOUR_P)
        seconds = hour * 3600 + rng.integers(0, 3600, n)

        frames.append(pd.DataFrame({
            "store_id": s.store_id,
            "customer_id": cust_ids[chosen],
            "placed_ts": day + pd.to_timedelta(seconds, unit="s"),
        }))

orders_raw = pd.concat(frames, ignore_index=True).sort_values("placed_ts").reset_index(drop=True)
orders_raw.insert(0, "order_id", [f"ORD{i:07d}" for i in range(1, len(orders_raw) + 1)])

In [0]:
#Basket size, promise time, and the generator's private plan

n_items = np.minimum(1 + rng.poisson(3.0, len(orders_raw)), 12)   # about 4 lines per order
orders_raw["promised_minutes"] = np.where(n_items <= 4, 10, 15)   # bigger baskets get a longer promise

# n_items is the generator's private knowledge (not in the raw feed); module 3 uses it
order_plan = pd.DataFrame({"order_id": orders_raw["order_id"], "n_items": n_items})

orders = orders_raw[["order_id", "customer_id", "store_id", "placed_ts", "promised_minutes"]]
assert orders["order_id"].is_unique
print(len(orders), "orders")

In [0]:
#save to bronze

save_bronze(orders, "orders")

In [0]:
%sql
--demand curve check

SELECT HOUR(placed_ts) AS hr, COUNT(*) AS orders
FROM workspace.bronze.orders
GROUP BY 1 ORDER BY 1;

In [0]:
%sql
--rain effect check
SELECT c.is_rainy,
       COUNT(DISTINCT c.date) AS days,
       ROUND(COUNT(*) / COUNT(DISTINCT c.date)) AS avg_orders_per_day
FROM workspace.bronze.orders o
JOIN workspace.bronze.daily_context c ON DATE(o.placed_ts) = c.date
GROUP BY c.is_rainy;